In [48]:
"""
Q1.1 读取附件1并合并为长表
输出: data/hourly_long.parquet  (columns: i, h, p)
"""
import pandas as pd
from pathlib import Path

# Jupyter 环境不使用 __file__，直接定义项目根目录
PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
SRC = PROJECT_ROOT / "A题" / "附件1.xlsx"
OUT = PROJECT_ROOT / "Q1输出" / "data"
OUT.mkdir(parents=True, exist_ok=True)

xl = pd.ExcelFile(SRC, engine="openpyxl")
print("Sheets:", xl.sheet_names)


Sheets: ['A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7', 'A_8', 'A_9', 'A_10']


In [49]:

frames = []
for sn in xl.sheet_names:
    df = xl.parse(sn)
    # 规范列名
    df.columns = [c.strip().lower() for c in df.columns]
    assert set(df.columns) >= {"time", "per"}, f"{sn} columns: {df.columns}"
    # 提取编号 i
    i = int(sn.split("_")[1])
    df = df.rename(columns={"time": "h", "per": "p"})[["h", "p"]]
    df["i"] = i
    df["h"] = pd.to_datetime(df["h"])
    df["p"] = pd.to_numeric(df["p"], errors="coerce")
    frames.append(df)
    print(f"  A_{i}: rows={len(df)}, p>100 count={(df['p']>100).sum()}, "
          f"p<0 count={(df['p']<0).sum()}, NaN={df['p'].isna().sum()}, "
          f"range=[{df['p'].min():.2f}, {df['p'].max():.2f}]")

long = pd.concat(frames, ignore_index=True).sort_values(["i", "h"]).reset_index(drop=True)
print(f"\nTotal rows: {len(long)}, filters: {long['i'].nunique()}")
print(f"Time span: {long['h'].min()} .. {long['h'].max()}")

# 保存
long.to_parquet(OUT / "hourly_long.parquet", index=False) if False else None
long.to_csv(OUT / "hourly_long.csv", index=False)
print(f"Saved: {OUT/'hourly_long.csv'}")


  A_1: rows=10839, p>100 count=3700, p<0 count=0, NaN=457, range=[38.64, 140.37]
  A_2: rows=11125, p>100 count=76, p<0 count=0, NaN=406, range=[28.06, 121.72]
  A_3: rows=11921, p>100 count=3759, p<0 count=0, NaN=409, range=[51.50, 154.57]
  A_4: rows=11675, p>100 count=4645, p<0 count=0, NaN=445, range=[31.56, 143.75]
  A_5: rows=12162, p>100 count=5561, p<0 count=0, NaN=458, range=[34.53, 149.81]
  A_6: rows=11806, p>100 count=1203, p<0 count=0, NaN=462, range=[28.65, 155.92]
  A_7: rows=11976, p>100 count=2276, p<0 count=0, NaN=456, range=[39.41, 158.57]
  A_8: rows=11507, p>100 count=4325, p<0 count=0, NaN=418, range=[37.63, 165.15]
  A_9: rows=11223, p>100 count=1208, p<0 count=0, NaN=455, range=[44.19, 139.39]
  A_10: rows=10743, p>100 count=3821, p<0 count=0, NaN=379, range=[26.43, 180.00]

Total rows: 114977, filters: 10
Time span: 2024-04-03 01:00:05 .. 2026-04-11 18:00:00
Saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q1输出\data\hourly_long.csv


In [50]:

# 简要统计
stats = long.groupby("i")["p"].agg(["count", "min", "max", "mean", "median"]).round(2)
stats["nan_count"] = long.groupby("i")["p"].apply(lambda s: s.isna().sum())
print("\nPer-filter summary:")
print(stats)
stats.to_csv(OUT / "per_filter_hourly_stats.csv")



Per-filter summary:
    count    min     max   mean  median  nan_count
i                                                 
1   10382  38.64  140.37  91.51   95.08        457
2   10719  28.06  121.72  73.73   73.79        406
3   11512  51.50  154.57  93.64   93.44        409
4   11230  31.56  143.75  92.00   96.74        445
5   11704  34.53  149.81  93.14   99.25        458
6   11344  28.65  155.92  74.03   73.32        462
7   11520  39.41  158.57  89.90   91.94        456
8   11089  37.63  165.15  95.54   95.03        418
9   10768  44.19  139.39  82.32   81.16        455
10  10364  26.43  180.00  94.85   94.50        379


In [51]:
"""
Q1.2 日聚合：y_{i,d}=median{p_i(h):h in d}, 保留 n_{i,d}
      n_{i,d}<12 时 y 置为 NaN
输出: data/daily_long.csv (columns: i, d, y, n, y_raw)
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
hourly = pd.read_csv(ROOT / "data/hourly_long.csv", parse_dates=["h"])
print(f"Loaded {len(hourly)} hourly rows")


Loaded 114977 hourly rows


In [52]:

hourly["d"] = hourly["h"].dt.normalize()  # 当天 00:00

# 丢掉 p=NaN 的小时，再做 median / count
valid = hourly.dropna(subset=["p"])
daily = (valid.groupby(["i", "d"])
              .agg(y_raw=("p", "median"), n=("p", "size"))
              .reset_index())

N_MIN = 12
daily["y"] = daily["y_raw"].where(daily["n"] >= N_MIN, np.nan)

# 生成完整日历 (i × all days)
all_days = pd.date_range(daily["d"].min(), daily["d"].max(), freq="D")
idx = pd.MultiIndex.from_product([sorted(daily["i"].unique()), all_days],
                                  names=["i", "d"])
daily = (daily.set_index(["i", "d"])
              .reindex(idx)
              .reset_index())


In [53]:

print("\nDaily median preview:")
daily.head(20)



Daily median preview:


,i,d,y_raw,n,y
0,1,2024-04-03,91.470808,21.0,91.470808
1,1,2024-04-04,91.904416,14.0,91.904416
2,1,2024-04-05,90.867819,16.0,90.867819
3,1,2024-04-06,90.904580,10.0,NaN
4,1,2024-04-07,NaN,NaN,NaN
5,1,2024-04-08,NaN,NaN,NaN
6,1,2024-04-09,90.592063,13.0,90.592063
7,1,2024-04-10,90.130523,21.0,90.130523
8,1,2024-04-11,90.250727,11.0,NaN
9,1,2024-04-12,89.485102,22.0,89.485102


In [54]:
daily["n"] = daily["n"].fillna(0).astype(int)

print(f"Daily long table: {len(daily)} rows  ({daily['i'].nunique()} filters × {len(all_days)} days)")
print(f"NaN y count: {daily['y'].isna().sum()}  "
      f"({100*daily['y'].isna().mean():.1f}% of rows)")
print(f"Rows with n<12 (masked): {((daily['n']>0)&(daily['n']<12)).sum()}")

# 每台统计
summary = (daily.groupby("i")
                .agg(days_total=("d", "size"),
                     days_valid=("y", lambda s: s.notna().sum()),
                     days_n_zero=("n", lambda s: (s==0).sum()),
                     days_n_lt12=("n", lambda s: ((s>0)&(s<12)).sum()),
                     y_min=("y", "min"),
                     y_max=("y", "max"),
                     y_mean=("y", "mean"))
                .round(2))
summary["coverage_%"] = (100 * summary["days_valid"] / summary["days_total"]).round(1)
print("\nPer-filter daily summary:")
print(summary)

daily.to_csv(ROOT / "data/daily_long.csv", index=False)
summary.to_csv(ROOT / "data/per_filter_daily_summary.csv")
print(f"\nSaved: {ROOT/'data/daily_long.csv'}")


Daily long table: 7390 rows  (10 filters × 739 days)
NaN y count: 2344  (31.7% of rows)
Rows with n<12 (masked): 1890

Per-filter daily summary:
    days_total  days_valid  days_n_zero  days_n_lt12  y_min   y_max  y_mean  \
i                                                                             
1          739         477           49          213  38.97  113.77   90.90   
2          739         490           43          206  43.97  100.70   74.28   
3          739         525           44          170  60.44  121.70   93.72   
4          739         518           56          165  38.23  125.43   91.83   
5          739         525           39          175  38.29  119.14   93.74   
6          739         514           51          174  29.15  123.34   73.90   
7          739         523           43          173  49.39  121.60   90.65   
8          739         503           43          193  59.83  131.21   96.17   
9          739         488           49          202  52.08  106.

In [55]:
"""
Q1.3 缺失与异常检测
 - 连续缺测段统计
 - 基于 IQR*1.5 与 3*MAD 两版异常检测（稳健）
 - 作图：每台的时间序列 + 异常点标记 + 维护事件竖线
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
daily = pd.read_csv(ROOT / "data/daily_long.csv", parse_dates=["d"])


In [56]:

# ---------- 连续缺测段 ----------
def longest_gap(sub):
    # sub 按时间升序；返回最长连续 y=NaN 的长度
    isnan = sub["y"].isna().values
    best = cur = 0
    for x in isnan:
        cur = cur + 1 if x else 0
        best = max(best, cur)
    return best

gap = (daily.sort_values(["i", "d"])
            .groupby("i").apply(longest_gap, include_groups=False)
            .rename("longest_gap_days"))
print("Longest consecutive NaN gap per filter:")
print(gap.to_string())


Longest consecutive NaN gap per filter:
i
1     83
2     82
3     82
4     82
5     82
6     82
7     82
8     82
9     82
10    82


In [57]:

# ---------- 异常检测（IQR 1.5 与 3×MAD 两版） ----------
def detect_outlier(sub):
    y = sub["y"].dropna()
    q1, q3 = y.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo1, hi1 = q1 - 1.5*iqr, q3 + 1.5*iqr
    med = y.median()
    mad = (y - med).abs().median()
    lo2, hi2 = med - 3*1.4826*mad, med + 3*1.4826*mad
    return pd.Series(dict(lo_iqr=lo1, hi_iqr=hi1, lo_mad=lo2, hi_mad=hi2,
                          n_outlier_iqr=((sub["y"]<lo1)|(sub["y"]>hi1)).sum(),
                          n_outlier_mad=((sub["y"]<lo2)|(sub["y"]>hi2)).sum()))

thr = daily.groupby("i").apply(detect_outlier, include_groups=False)
print("\nOutlier thresholds & counts per filter:")
print(thr.round(2))
thr.to_csv(ROOT / "data/outlier_thresholds.csv")



Outlier thresholds & counts per filter:
    lo_iqr  hi_iqr  lo_mad  hi_mad  n_outlier_iqr  n_outlier_mad
i                                                               
1    54.07  131.77   57.20  132.95           19.0           25.0
2    41.42  108.31   37.11  111.18            0.0            0.0
3    51.91  132.44   47.32  140.05            0.0            0.0
4    49.68  136.46   53.49  139.54           37.0           37.0
5    60.14  133.93   60.25  138.37           41.0           42.0
6     9.39  139.57   -4.14  149.87            0.0            0.0
7    65.74  118.39   63.05  121.40           33.0           20.0
8    53.21  138.42   48.12  142.69            0.0            0.0
9    49.13  115.99   42.91  119.83            0.0            0.0
10   28.58  164.06   21.98  168.28            0.0            0.0


In [58]:

# 把异常标签回填到 daily
daily = daily.merge(thr[["lo_iqr", "hi_iqr"]].reset_index(), on="i")
daily["is_outlier"] = ((daily["y"] < daily["lo_iqr"]) | (daily["y"] > daily["hi_iqr"]))


In [59]:

# ---------- 作图（每台一幅） ----------
# 先载入维护事件用于竖线
mnt = pd.read_excel(PROJECT_ROOT / "A题" / "附件2.xlsx",
                    sheet_name=0, engine="openpyxl")
mnt.columns = [c.strip() for c in mnt.columns]
mnt = mnt.rename(columns={"编号": "id_str", "日期": "d", "维护类型": "q"})
mnt["i"] = mnt["id_str"].str.replace("A", "", regex=False).astype(int)
mnt["d"] = pd.to_datetime(mnt["d"])
mnt["q"] = mnt["q"].map({"小维护": "s", "中维护": "m", "大维护": "l"})
print(f"\nMaintenance records: {len(mnt)};  by type: "
      f"{mnt['q'].value_counts().to_dict()}")
print(f"Per-filter large maintenance count:\n"
      f"{mnt[mnt['q']=='l'].groupby('i').size().reindex(range(1,11), fill_value=0).to_string()}")

fig_dir = ROOT / "figures"
fig_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(5, 2, figsize=(16, 14), sharex=True)
for ax, i in zip(axes.ravel(), range(1, 11)):
    sub = daily[daily["i"] == i]
    ax.plot(sub["d"], sub["y"], ".", ms=2, c="steelblue", alpha=0.6)
    out = sub[sub["is_outlier"].fillna(False)]
    ax.plot(out["d"], out["y"], "x", c="red", ms=4, label=f"outlier ({len(out)})")
    # 维护竖线
    ms = mnt[mnt["i"] == i]
    for _, r in ms.iterrows():
        c = {"s": "grey", "m": "orange", "l": "darkgreen"}[r["q"]]
        ax.axvline(r["d"], c=c, alpha=0.35, lw=0.8)
    ax.axhline(37, c="red", ls="--", lw=0.5)
    ax.set_title(f"A{i} (valid={sub['y'].notna().sum()}d)", fontsize=9)
    ax.set_ylim(20, 200)
    ax.legend(loc="upper right", fontsize=7)
fig.suptitle("Daily permeability y_{i,d} (blue dots), outliers (red x), "
             "maintenance events (orange=M, green=L)", fontsize=11)
fig.tight_layout()
fig.savefig(fig_dir / "fig1_daily_series_outliers.png", dpi=130)
plt.close(fig)
print(f"Figure saved: {fig_dir/'fig1_daily_series_outliers.png'}")

daily.to_csv(ROOT / "data/daily_long.csv", index=False)
print("Daily table updated with outlier flags.")



Maintenance records: 127;  by type: {'m': 110, 'l': 17}
Per-filter large maintenance count:
i
1     2
2     3
3     2
4     0
5     1
6     2
7     2
8     0
9     2
10    3
Figure saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q1输出\figures\fig1_daily_series_outliers.png
Daily table updated with outlier flags.


In [60]:
"""
Q1.4 读取附件2 并构造维护协变量
输出:
  data/maintenance_events.csv  (i, d, q)
  data/daily_with_vars.csv     (i, d, y, n, is_outlier, u_m, u_l,
                                H_m7, H_l7, A_m, A_l, sin, cos, t)
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"


In [61]:

# ---------- 维护事件 ----------
mnt = pd.read_excel(PROJECT_ROOT / "A题" / "附件2.xlsx",
                    sheet_name=0, engine="openpyxl")
mnt.columns = [c.strip() for c in mnt.columns]
mnt = mnt.rename(columns={"编号": "id_str", "日期": "d", "维护类型": "q"})
mnt["i"] = mnt["id_str"].str.replace("A", "", regex=False).astype(int)
mnt["d"] = pd.to_datetime(mnt["d"]).dt.normalize()
mnt["q"] = mnt["q"].map({"小维护": "s", "中维护": "m", "大维护": "l"})
mnt = mnt[["i", "d", "q"]].sort_values(["i", "d"]).reset_index(drop=True)
print(f"Maintenance events: {len(mnt)};  distribution:\n{mnt['q'].value_counts()}")
mnt.to_csv(ROOT / "data/maintenance_events.csv", index=False)


Maintenance events: 127;  distribution:
q
m    110
l     17
Name: count, dtype: int64


In [62]:

# ---------- 日表 ----------
daily = pd.read_csv(ROOT / "data/daily_long.csv", parse_dates=["d"])
daily = daily.sort_values(["i", "d"]).reset_index(drop=True)


In [63]:

# ---------- 构造 u^q, H^{q,7}, A^q ----------
W = 7

def build_for_filter(i, sub):
    sub = sub.sort_values("d").reset_index(drop=True).copy()
    events = mnt[mnt["i"] == i].sort_values("d")
    # u^q: 当天是否维护
    sub["u_m"] = sub["d"].isin(events[events["q"] == "m"]["d"]).astype(int)
    sub["u_l"] = sub["d"].isin(events[events["q"] == "l"]["d"]).astype(int)

    # H^{q,7}: 最近一次 q 类维护后 1..W 天窗口（不含当天 τ）
    def window_flag(q):
        flag = np.zeros(len(sub), dtype=int)
        days = events[events["q"] == q]["d"].tolist()
        d_arr = sub["d"].values
        for tau in days:
            mask = (d_arr > np.datetime64(tau)) & (d_arr <= np.datetime64(tau + pd.Timedelta(days=W)))
            flag[mask] = 1
        return flag

    sub["H_m7"] = window_flag("m")
    sub["H_l7"] = window_flag("l")

    # A^q: 距上次 q 类维护的天数（维护当天 = 0；若尚未有过维护，则为 NaN）
    def days_since(q):
        last = pd.NaT
        out = np.full(len(sub), np.nan)
        ev_days = set(events[events["q"] == q]["d"].tolist())
        for j, d in enumerate(sub["d"]):
            if d in ev_days:
                last = d
            if pd.notna(last):
                out[j] = (d - last).days
        return out

    sub["A_m"] = days_since("m")
    sub["A_l"] = days_since("l")
    return sub

out = []
for i, sub in daily.groupby("i"):
    out.append(build_for_filter(i, sub))
daily2 = pd.concat(out, ignore_index=True)
daily2.head(298)


,i,d,y_raw,n,y,lo_iqr,hi_iqr,is_outlier,u_m,u_l,H_m7,H_l7,A_m,A_l
0,1,2024-04-03,91.470808,21,91.470808,54.065501,131.76716,False,0,0,0,0,NaN,NaN
1,1,2024-04-04,91.904416,14,91.904416,54.065501,131.76716,False,0,0,0,0,NaN,NaN
2,1,2024-04-05,90.867819,16,90.867819,54.065501,131.76716,False,0,0,0,0,NaN,NaN
3,1,2024-04-06,90.904580,10,NaN,54.065501,131.76716,False,0,0,0,0,NaN,NaN
4,1,2024-04-07,NaN,0,NaN,54.065501,131.76716,False,0,0,0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,1,2025-01-21,NaN,0,NaN,54.065501,131.76716,False,1,0,0,0,0.0,50.0
294,1,2025-01-22,95.950259,14,95.950259,54.065501,131.76716,False,0,0,1,0,1.0,51.0
295,1,2025-01-23,95.613424,20,95.613424,54.065501,131.76716,False,0,0,1,0,2.0,52.0
296,1,2025-01-24,95.388092,21,95.388092,54.065501,131.76716,False,0,0,1,0,3.0,53.0


In [64]:

# ---------- 时间/季节协变量 ----------
anchor = daily2["d"].min()
daily2["t"] = (daily2["d"] - anchor).dt.days.astype(float)   # 天数
T = 365.25
daily2["sin1"] = np.sin(2*np.pi*daily2["t"]/T)
daily2["cos1"] = np.cos(2*np.pi*daily2["t"]/T)


In [65]:

# ---------- 季节标签（便于分桶） ----------
m = daily2["d"].dt.month
daily2["season"] = np.select(
    [m.isin([3,4,5]), m.isin([6,7,8]), m.isin([9,10,11])],
    ["春", "夏", "秋"], default="冬"
)

print(f"\nDaily enriched table: {len(daily2)} rows")



Daily enriched table: 7390 rows


In [66]:
print("Maintenance flag counts:")
print(daily2[["u_m", "u_l", "H_m7", "H_l7"]].sum())
print(f"A_m range: {daily2['A_m'].min()} .. {daily2['A_m'].max()};  "
      f"A_l range: {daily2['A_l'].min()} .. {daily2['A_l'].max()}")


daily2.to_csv(ROOT / "data/daily_with_vars.csv", index=False)
print(f"\nSaved: {ROOT/'data/daily_with_vars.csv'}")


Maintenance flag counts:
u_m     110
u_l      17
H_m7    769
H_l7    119
dtype: int64
A_m range: 0.0 .. 233.0;  A_l range: 0.0 .. 567.0

Saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q1输出\data\daily_with_vars.csv


In [69]:
"""
Q1.5 STL-lite 分解（无 statsmodels）
 分量：
   趋势 T_d = 90 天中心化滚动中位（对 NaN 容忍）
   季节 S_d = 去趋势后按 day-of-year 的平均
   残差 R_d = y_d - T_d - S_d
输出:
  data/stl_components.csv
  figures/fig2_stl_decomposition.png
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
df = pd.read_csv(ROOT / "data/daily_with_vars.csv", parse_dates=["d"])


In [70]:
def rolling_median_nanaware(y, win=91):
    s = pd.Series(y)
    return s.rolling(window=win, center=True, min_periods=max(15, win//4)).median().values


In [71]:

def doy_mean(detr, doy):
    tab = pd.DataFrame({"doy": doy, "v": detr}).groupby("doy")["v"].mean()
    return tab


In [72]:

parts = []
for i, sub in df.groupby("i"):
    sub = sub.sort_values("d").reset_index(drop=True).copy()
    sub["trend"] = rolling_median_nanaware(sub["y"].values, 91)
    sub["detrended"] = sub["y"] - sub["trend"]
    doy = sub["d"].dt.dayofyear
    seas_table = doy_mean(sub["detrended"].values, doy.values)
    # 居中到 0
    seas_table = seas_table - seas_table.mean()
    sub["season"] = doy.map(seas_table)
    sub["resid"] = sub["y"] - sub["trend"] - sub["season"]
    parts.append(sub)

stl = pd.concat(parts, ignore_index=True)
stl.to_csv(ROOT / "data/stl_components.csv", index=False)


In [82]:
stl.head(5)

,i,d,y_raw,n,y,lo_iqr,hi_iqr,is_outlier,u_m,u_l,...,H_l7,A_m,A_l,t,sin1,cos1,season,trend,detrended,resid
0,1,2024-04-03,91.470808,21,91.470808,54.065501,131.76716,False,0,0,...,0,NaN,NaN,0.0,0.000000,1.000000,3.456481,88.232690,3.238119,-0.218362
1,1,2024-04-04,91.904416,14,91.904416,54.065501,131.76716,False,0,0,...,0,NaN,NaN,1.0,0.017202,0.999852,3.417911,88.232690,3.671727,0.253816
2,1,2024-04-05,90.867819,16,90.867819,54.065501,131.76716,False,0,0,...,0,NaN,NaN,2.0,0.034398,0.999408,2.907563,88.250685,2.617134,-0.290428
3,1,2024-04-06,90.904580,10,NaN,54.065501,131.76716,False,0,0,...,0,NaN,NaN,3.0,0.051584,0.998669,1.895869,88.268680,NaN,NaN
4,1,2024-04-07,NaN,0,NaN,54.065501,131.76716,False,0,0,...,0,NaN,NaN,4.0,0.068755,0.997634,1.431418,88.719782,NaN,NaN


In [83]:

# 作图：每台 4 子图（原始/趋势/季节/残差）
fig, axes = plt.subplots(10, 4, figsize=(18, 24), sharex=True)
for row, i in enumerate(range(1, 11)):
    sub = stl[stl["i"] == i].sort_values("d")
    axes[row, 0].plot(sub["d"], sub["y"], ".", ms=1.5, c="steelblue"); axes[row, 0].set_ylabel(f"A{i}", fontsize=9)
    axes[row, 1].plot(sub["d"], sub["trend"], "-", c="darkorange", lw=1)
    axes[row, 2].plot(sub["d"], sub["season"], "-", c="seagreen", lw=0.7)
    axes[row, 3].plot(sub["d"], sub["resid"], ".", ms=1.2, c="grey")
    for j in range(4): axes[row, j].grid(alpha=0.3)
    if row == 0:
        for j, t in enumerate(["observed y", "trend (91d rolling median)",
                                "seasonal (DOY mean)", "residual"]):
            axes[row, j].set_title(t, fontsize=10)
fig.suptitle("Q1.5  STL-lite decomposition per filter", fontsize=14, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(ROOT / "figures/fig2_stl_decomposition.png", dpi=120)
plt.close(fig)
print(f"Saved: {ROOT/'figures/fig2_stl_decomposition.png'}")


Saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q1输出\figures\fig2_stl_decomposition.png


In [84]:

# 指标汇总：线性趋势斜率（稳健）、季节幅度、残差 std
from numpy.polynomial import polynomial as P
summary_rows = []
for i, sub in stl.groupby("i"):
    sub = sub.dropna(subset=["y", "trend"])
    t = sub["t"].values
    tr = sub["trend"].values
    # 以趋势为被解释变量做 OLS 求斜率（天per天）
    mask = ~np.isnan(tr)
    if mask.sum() > 30:
        slope, intercept = np.polyfit(t[mask], tr[mask], 1)
    else:
        slope, intercept = np.nan, np.nan
    season_amp = (sub["season"].max() - sub["season"].min()) if sub["season"].notna().any() else np.nan
    resid_std = sub["resid"].std()
    summary_rows.append(dict(i=i,
                             trend_slope_per_day=slope,
                             trend_slope_per_year=slope*365.25,
                             season_amp=season_amp,
                             resid_std=resid_std))
summary = pd.DataFrame(summary_rows).round(3)
summary.to_csv(ROOT / "data/stl_summary.csv", index=False)
print("STL summary:")
print(summary.to_string(index=False))


STL summary:
 i  trend_slope_per_day  trend_slope_per_year  season_amp  resid_std
 1               -0.024                -8.696      32.608      2.618
 2               -0.020                -7.392      21.840      2.998
 3               -0.029               -10.599      25.818      3.789
 4                0.055                19.948      31.547      3.685
 5               -0.076               -27.672      27.461      5.025
 6                0.092                33.490      40.282      2.953
 7               -0.019                -6.864      29.432      4.649
 8               -0.070               -25.587      30.429      2.332
 9               -0.022                -8.017      33.829      3.559
10               -0.113               -41.447      29.669      2.618


In [87]:
"""
Q1.6 Q1 回归模型（不含 γ_q，按既定口径）
   y_{i,d} = α_i + β_i·t + a·sin + b·cos + η_m·H_m7 + η_l·H_l7 + ρ_m·A_m + ρ_l·A_l + ε
  - 用 filter fixed effects (dummy)
  - 允许 β_i 按台不同（交互项 i×t）
  - A_m / A_l 在无历史维护时填 0（作为参考水平），同时加 has_m / has_l 指示变量
  - 双版本：普通 OLS 与 WLS (权重=n_{i,d})
输出：
  tables/reg_summary_ols.csv
  tables/reg_summary_wls.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
df = pd.read_csv(ROOT / "data/daily_with_vars.csv", parse_dates=["d"])


In [88]:

# 准备变量
df = df.dropna(subset=["y"]).copy()
df["has_m"] = df["A_m"].notna().astype(int)
df["has_l"] = df["A_l"].notna().astype(int)
df["A_m_f"] = df["A_m"].fillna(0)
df["A_l_f"] = df["A_l"].fillna(0)


In [89]:

# 构造设计矩阵 X
# 列： intercept (global), 9个过滤器dummy (A2..A10, A1为基准), 10个 β_i·t 交互,
#      sin1, cos1, H_m7, H_l7, A_m_f, A_l_f, has_m, has_l
filters = sorted(df["i"].unique())
ref = filters[0]

def build_X(df, add_filter_specific_trend=True):
    parts = [np.ones(len(df))]
    names = ["const"]
    # 过滤器 FE
    for i in filters[1:]:
        parts.append((df["i"] == i).astype(float).values)
        names.append(f"I(i={i})")
    # 趋势 β_i · t 交互（每台一个）
    if add_filter_specific_trend:
        for i in filters:
            parts.append(((df["i"] == i).astype(float) * df["t"]).values)
            names.append(f"t_i{i}")
    else:
        parts.append(df["t"].values); names.append("t")
    # 季节
    parts.append(df["sin1"].values); names.append("sin1")
    parts.append(df["cos1"].values); names.append("cos1")
    # 维护
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        parts.append(df[col].values.astype(float)); names.append(col)
    X = np.column_stack(parts)
    return X, names


In [90]:

def ols(X, y, w=None):
    """最小二乘 + HC0 稳健标准误（可加权）。"""
    if w is None:
        Wy = y
        WX = X
    else:
        sw = np.sqrt(w)
        Wy = y * sw
        WX = X * sw[:, None]
    beta, *_ = np.linalg.lstsq(WX, Wy, rcond=None)
    # 残差
    resid = y - X @ beta
    n, k = X.shape
    # 普通方差
    if w is None:
        sigma2 = (resid**2).sum() / (n - k)
        XtX_inv = np.linalg.inv(X.T @ X)
        var_beta = sigma2 * XtX_inv
    else:
        # WLS: var = (X'WX)^-1 X'W diag(r^2) W X (X'WX)^-1  (HC0 variant)
        W = np.diag(w)
        XtWX_inv = np.linalg.inv(X.T @ W @ X)
        meat = X.T @ W @ np.diag(resid**2) @ W @ X
        var_beta = XtWX_inv @ meat @ XtWX_inv
    se = np.sqrt(np.diag(var_beta))
    tstat = beta / np.where(se == 0, np.nan, se)
    # R^2
    ss_res = (resid**2).sum()
    ss_tot = ((y - y.mean())**2).sum()
    r2 = 1 - ss_res / ss_tot
    return beta, se, tstat, r2, resid


In [91]:

X, names = build_X(df, add_filter_specific_trend=True)
y = df["y"].values
w = df["n"].values.astype(float)

b_ols, se_ols, t_ols, r2_ols, res_ols = ols(X, y, w=None)
b_wls, se_wls, t_wls, r2_wls, res_wls = ols(X, y, w=w)
print(f"OLS  R² = {r2_ols:.4f},  n = {len(y)},  k = {X.shape[1]}")
print(f"WLS  R² = {r2_wls:.4f}")

tab = pd.DataFrame({
    "var": names,
    "beta_OLS": b_ols,
    "SE_OLS": se_ols,
    "t_OLS": t_ols,
    "beta_WLS": b_wls,
    "SE_WLS": se_wls,
    "t_WLS": t_wls,
}).round(4)
tab.to_csv(ROOT / "tables/reg_summary_full.csv", index=False)


OLS  R² = 0.8385,  n = 5046,  k = 28
WLS  R² = 0.8384


In [92]:

# 关键系数摘要
key = tab[tab["var"].isin(["sin1", "cos1", "H_m7", "H_l7", "A_m_f", "A_l_f"])].copy()
key["ampl(sin+cos)"] = np.where(key["var"] == "sin1",
                                 np.sqrt(b_ols[names.index("sin1")]**2 +
                                         b_ols[names.index("cos1")]**2), np.nan)
print("\nKey coefficients:")
print(tab[tab["var"].isin(["const","sin1","cos1","H_m7","H_l7","A_m_f","A_l_f","has_m","has_l"])].to_string(index=False))



Key coefficients:
  var  beta_OLS  SE_OLS    t_OLS  beta_WLS  SE_WLS    t_WLS
const   91.6917  0.8676 105.6859   91.1999  0.7622 119.6512
 sin1   11.3030  0.1606  70.3961   11.3047  0.1546  73.1064
 cos1   -6.7992  0.1767 -38.4830   -6.7977  0.1876 -36.2424
 H_m7    1.9106  0.4072   4.6916    1.8877  0.4164   4.5331
 H_l7    2.2523  0.9116   2.4706    2.2404  0.8168   2.7428
A_m_f   -0.2183  0.0054 -40.3240   -0.2181  0.0055 -39.4829
A_l_f   -0.0301  0.0029 -10.4406   -0.0301  0.0029 -10.4181
has_m   13.3625  0.5580  23.9473   13.5241  0.7786  17.3689
has_l   14.6784  0.5043  29.1057   14.5436  0.4760  30.5556


In [93]:

# 各台趋势斜率 β_i (天/天) 与 年化
trend_rows = []
for i in filters:
    j = names.index(f"t_i{i}")
    trend_rows.append(dict(i=i, beta_per_day=b_ols[j], SE=se_ols[j],
                           beta_per_year=b_ols[j]*365.25, t=t_ols[j]))
trend = pd.DataFrame(trend_rows).round(4)
print("\nFilter-specific trend β_i (OLS):")
print(trend.to_string(index=False))
trend.to_csv(ROOT / "tables/trend_per_filter.csv", index=False)



Filter-specific trend β_i (OLS):
 i  beta_per_day     SE  beta_per_year        t
 1       -0.0337 0.0024       -12.3150 -14.3008
 2       -0.0096 0.0021        -3.5092  -4.5256
 3       -0.0415 0.0021       -15.1400 -19.3150
 4        0.0523 0.0019        19.1100  27.6467
 5       -0.0797 0.0030       -29.1114 -26.5910
 6        0.0776 0.0025        28.3305  30.9167
 7       -0.0435 0.0024       -15.8753 -18.1502
 8       -0.0767 0.0019       -28.0011 -40.3310
 9       -0.0434 0.0024       -15.8467 -18.4431
10       -0.1243 0.0021       -45.3892 -59.1750


In [94]:

# 季节性幅度
sin_idx = names.index("sin1"); cos_idx = names.index("cos1")
amp = np.sqrt(b_ols[sin_idx]**2 + b_ols[cos_idx]**2)
phase = np.degrees(np.arctan2(b_ols[sin_idx], b_ols[cos_idx]))
print(f"\nSeasonal amplitude = {amp:.3f},  phase = {phase:.1f}°  "
      f"(peak ~ DOY {(90-phase)%360/360*365:.0f})")



Seasonal amplitude = 13.190,  phase = 121.0°  (peak ~ DOY 334)


In [95]:

# 保存残差
df["resid_ols"] = res_ols
df["resid_wls"] = res_wls
df[["i","d","y","n","resid_ols","resid_wls"]].to_csv(ROOT / "data/regression_residuals.csv", index=False)
print(f"\nSaved: tables/reg_summary_full.csv,  tables/trend_per_filter.csv,  data/regression_residuals.csv")



Saved: tables/reg_summary_full.csv,  tables/trend_per_filter.csv,  data/regression_residuals.csv


In [3]:
"""
Q1.7 维护效果指标 R^{q,7}_{i,k} = y_post7 - y_pre7 (excluding τ day itself)
  - pre = 前 7 天 [τ-7, τ-1]
  - post= 后 7 天 [τ+1, τ+7]
  - 两边各需至少 3 个有效 y 才计算
  - 按季节（春/夏/秋/冬）与类型（m/l）分桶
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
daily = pd.read_csv(ROOT / "data/daily_long.csv", parse_dates=["d"])
mnt = pd.read_csv(ROOT / "data/maintenance_events.csv", parse_dates=["d"])


In [4]:

def season_of(d):
    m = d.month
    if m in (3,4,5): return "春"
    if m in (6,7,8): return "夏"
    if m in (9,10,11): return "秋"
    return "冬"

# 以 (i, d) 为键的 y 字典
y_lookup = {(r.i, r.d): r.y for r in daily.itertuples()}

rows = []
for ev in mnt.itertuples():
    i, tau, q = ev.i, ev.d, ev.q
    pre_days  = [tau - pd.Timedelta(days=j) for j in range(1, 8)]
    post_days = [tau + pd.Timedelta(days=j) for j in range(1, 8)]
    y_pre  = [y_lookup.get((i, dd), np.nan) for dd in pre_days]
    y_post = [y_lookup.get((i, dd), np.nan) for dd in post_days]
    y_pre  = np.array(y_pre, dtype=float)
    y_post = np.array(y_post, dtype=float)
    n_pre  = np.isfinite(y_pre).sum()
    n_post = np.isfinite(y_post).sum()
    if n_pre < 3 or n_post < 3:
        continue
    R = np.nanmean(y_post) - np.nanmean(y_pre)
    rows.append(dict(i=i, d=tau, q=q, season=season_of(tau),
                     y_pre=np.nanmean(y_pre), y_post=np.nanmean(y_post),
                     n_pre=n_pre, n_post=n_post, R=R))

R = pd.DataFrame(rows)
print(f"R indicator computed on {len(R)} events (of {len(mnt)} total)")
R.to_csv(ROOT / "tables/R_events.csv", index=False)


R indicator computed on 94 events (of 127 total)


In [5]:

# 按类型汇总
by_q = R.groupby("q")["R"].agg(["count", "mean", "median", "std",
                                 lambda s: s.quantile(0.1),
                                 lambda s: s.quantile(0.9)]).round(2)
by_q.columns = ["n", "mean", "median", "std", "q10", "q90"]
print("\nR by maintenance type:")
print(by_q)
by_q.to_csv(ROOT / "tables/R_by_type.csv")
by_q.head(10)


R by maintenance type:
    n   mean  median   std   q10    q90
q                                      
l  13  12.81   13.99  6.13  4.33  17.82
m  81  16.46   16.88  9.94  2.61  27.67


,n,mean,median,std,q10,q90
q,,,,,,
l,13,12.81,13.99,6.13,4.33,17.82
m,81,16.46,16.88,9.94,2.61,27.67


In [6]:

# 按季节×类型汇总
by_qs = R.groupby(["q", "season"])["R"].agg(["count", "mean", "median", "std"]).round(2)
print("\nR by (type × season):")
print(by_qs)
by_qs.to_csv(ROOT / "tables/R_by_type_season.csv")
by_qs.head(10)


R by (type × season):
          count   mean  median    std
q season                             
l 冬           4  15.67   15.32   5.26
  夏           1   9.78    9.78    NaN
  春           3  15.69   15.12   2.04
  秋           5   9.41   12.04   7.66
m 冬          23  18.01   20.00  11.31
  夏          16  15.56   17.23   9.19
  春          17  18.66   15.94   9.61
  秋          25  14.12   14.83   9.24


count   mean  median    std
q season                             
l 冬           4  15.67   15.32   5.26
  夏           1   9.78    9.78    NaN
  春           3  15.69   15.12   2.04
  秋           5   9.41   12.04   7.66
m 冬          23  18.01   20.00  11.31
  夏          16  15.56   17.23   9.19
  春          17  18.66   15.94   9.61
  秋          25  14.12   14.83   9.24

In [9]:

# 按过滤器×类型
by_iq = R.groupby(["i", "q"])["R"].agg(["count", "mean"]).round(2)
print("\nR by (filter × type):")
print(by_iq.to_string())
by_iq.to_csv(ROOT / "tables/R_by_filter_type.csv")
by_iq.head(20)


R by (filter × type):
      count   mean
i  q              
1  l      2   8.23
   m      6  17.28
2  l      2  12.99
   m      7  15.88
3  l      2  12.37
   m      6  17.92
4  m     10  19.33
5  l      1   2.98
   m      9  15.40
6  l      2  19.81
   m      7  26.03
7  l      1  17.95
   m      9  12.27
8  m     10  13.45
9  l      1  15.12
   m      9  17.10
10 l      2  11.87
   m      8  12.24


count   mean
i  q              
1  l      2   8.23
   m      6  17.28
2  l      2  12.99
   m      7  15.88
3  l      2  12.37
   m      6  17.92
4  m     10  19.33
5  l      1   2.98
   m      9  15.40
6  l      2  19.81
   m      7  26.03
7  l      1  17.95
   m      9  12.27
8  m     10  13.45
9  l      1  15.12
   m      9  17.10
10 l      2  11.87
   m      8  12.24

In [10]:

# θ 阈值（大维护改善量的 10% 分位，用于 Q2 报废判据）
l_R = R[R["q"] == "l"]["R"]
if len(l_R) > 0:
    theta_10 = np.quantile(l_R, 0.10)
    print(f"\nθ (10% quantile of large maintenance improvement) = {theta_10:.2f}")
    print(f"Negative L-maintenance count: {(l_R < 0).sum()} / {len(l_R)}")



θ (10% quantile of large maintenance improvement) = 4.33
Negative L-maintenance count: 1 / 13


In [ ]:
"""
Q1.8 整合图表（EN titles to avoid CJK font issues）
  fig3  R boxplot by type×season
  fig4  trend β_i per filter
  fig5  fitted vs observed
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
R = pd.read_csv(ROOT / "tables/R_events.csv")
trend = pd.read_csv(ROOT / "tables/trend_per_filter.csv")
daily = pd.read_csv(ROOT / "data/daily_with_vars.csv", parse_dates=["d"])
resid = pd.read_csv(ROOT / "data/regression_residuals.csv", parse_dates=["d"])

# map Chinese season to English
SMAP = {"春": "Spring", "夏": "Summer", "秋": "Autumn", "冬": "Winter"}
R["season_en"] = R["season"].map(SMAP)

# --- Fig 3: R boxplot by type×season ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
seasons = ["Spring", "Summer", "Autumn", "Winter"]
for ax, q, lab in zip(axes, ["m", "l"], ["Medium maintenance", "Large maintenance"]):
    dat = [R[(R["q"] == q) & (R["season_en"] == s)]["R"].values for s in seasons]
    ax.boxplot(dat, labels=seasons, showmeans=True, meanline=True)
    ax.axhline(0, c="grey", lw=0.7, ls="--")
    ax.set_title(f"{lab}  (n={(R['q']==q).sum()})")
    ax.set_ylabel("R = mean(y_post7) - mean(y_pre7)")
    ax.grid(alpha=0.3)
fig.suptitle("Core indicator R^{q,7}: post - pre 7-day permeability delta by type x season",
             fontsize=11)
fig.tight_layout()
fig.savefig(ROOT / "figures/fig3_R_boxplot.png", dpi=130)
plt.close(fig)

# --- Fig 4: trend per filter ---
fig, ax = plt.subplots(figsize=(9, 4.5))
trend_sorted = trend.sort_values("beta_per_year")
colors = ["#d9534f" if v < 0 else "#5cb85c" for v in trend_sorted["beta_per_year"]]
bars = ax.barh([f"A{i}" for i in trend_sorted["i"]],
               trend_sorted["beta_per_year"], color=colors, edgecolor="k")
ax.axvline(0, c="k", lw=0.7)
ax.set_xlabel("Annualized trend slope  (permeability index / year)")
ax.set_title("Filter-specific natural aging slope (OLS estimate)")
for bar, v in zip(bars, trend_sorted["beta_per_year"]):
    ax.text(v, bar.get_y()+bar.get_height()/2,
            f" {v:+.1f}", va="center", fontsize=9,
            ha="left" if v >= 0 else "right")
ax.grid(alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(ROOT / "figures/fig4_trend_per_filter.png", dpi=130)
plt.close(fig)

# --- Fig 5: fitted vs observed ---
daily2 = daily.dropna(subset=["y"]).copy()
daily2 = daily2.merge(resid[["i","d","resid_ols"]], on=["i","d"])
daily2["y_hat"] = daily2["y"] - daily2["resid_ols"]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(daily2["y"], daily2["y_hat"], s=2, alpha=0.3, c="steelblue")
lim = [daily2["y"].min()-5, daily2["y"].max()+5]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Observed y")
ax.set_ylabel("Fitted y_hat")
r2 = 1 - (daily2["resid_ols"]**2).sum() / ((daily2["y"]-daily2["y"].mean())**2).sum()
ax.set_title(f"OLS fitted vs observed    R^2 = {r2:.3f},  n = {len(daily2)}")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ROOT / "figures/fig5_fit_vs_observed.png", dpi=130)
plt.close(fig)

print("Figures saved:")
for f in ["fig3_R_boxplot", "fig4_trend_per_filter", "fig5_fit_vs_observed"]:
    print("  ", ROOT / f"figures/{f}.png")

